In [1]:
!pip install hezar

In [2]:
from hezar.models import Model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import os
import yaml

In [ ]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = nn.functional.relu(x)
        return x

In [18]:
class CRNN(nn.Module):
    def __init__(self, n_channels, num_classes, map2seq_in_dim, map2seq_out_dim, rnn_dim):
        super(CRNN, self).__init__()
        # Define cnn as ModuleDict with the same indices as in model.pt
        self.cnn = nn.ModuleDict({
            '0': CNNBlock(n_channels, 64),
            '2': CNNBlock(64, 128),
            '4': CNNBlock(128, 256),
            '5': CNNBlock(256, 256),
            '7': CNNBlock(256, 512),
            '8': CNNBlock(512, 512),
            '10': CNNBlock(512, 512, kernel_size=2, padding=0)
        })
        self.map2seq = nn.Linear(map2seq_in_dim, map2seq_out_dim)
        self.rnn1 = nn.LSTM(map2seq_out_dim, rnn_dim, bidirectional=True, batch_first=True)
        self.rnn2 = nn.LSTM(rnn_dim * 2, rnn_dim, bidirectional=True, batch_first=True)
        self.classifier = nn.Linear(rnn_dim * 2, num_classes)

    def forward(self, x):
        # Apply CNN blocks in order with appropriate pooling
        for key in sorted(self.cnn.keys(), key=lambda k: int(k)):
            x = self.cnn[key](x)
            if key in ['0', '2']:
                x = nn.functional.max_pool2d(x, 2)  # (2, 2) pooling
            elif key in ['5', '8']:
                x = nn.functional.max_pool2d(x, (2, 1))  # (2, 1) pooling
        batch, channels, height, width = x.size()
        x = x.permute(0, 3, 1, 2).reshape(batch, width, -1)
        x = self.map2seq(x)
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x = self.classifier(x)
        return x

RuntimeError: Error(s) in loading state_dict for CRNN:
	Missing key(s) in state_dict: "cnn.0.weight", "cnn.0.bias", "cnn.1.weight", "cnn.1.bias", "cnn.1.running_mean", "cnn.1.running_var", "cnn.4.weight", "cnn.4.bias", "cnn.5.weight", "cnn.5.bias", "cnn.5.running_mean", "cnn.5.running_var", "cnn.8.weight", "cnn.8.bias", "cnn.9.weight", "cnn.9.bias", "cnn.9.running_mean", "cnn.9.running_var", "cnn.11.weight", "cnn.11.bias", "cnn.12.weight", "cnn.12.bias", "cnn.12.running_mean", "cnn.12.running_var", "cnn.15.weight", "cnn.15.bias", "cnn.16.weight", "cnn.16.bias", "cnn.16.running_mean", "cnn.16.running_var", "cnn.18.weight", "cnn.18.bias", "cnn.19.weight", "cnn.19.bias", "cnn.19.running_mean", "cnn.19.running_var", "cnn.22.weight", "cnn.22.bias", "cnn.23.weight", "cnn.23.bias", "cnn.23.running_mean", "cnn.23.running_var". 
	Unexpected key(s) in state_dict: "cnn.0.conv.weight", "cnn.0.conv.bias", "cnn.0.bn.weight", "cnn.0.bn.bias", "cnn.0.bn.running_mean", "cnn.0.bn.running_var", "cnn.0.bn

In [4]:
def preprocess_image(image_path):
    with open("/content/drive/MyDrive/crnn-fa-printed-96-long/preprocessor/image_processor_config.yaml", "r") as f:
        config = yaml.safe_load(f)

    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((32, 384)),
        transforms.Lambda(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT) if config["mirror"] else x),
        transforms.ToTensor(),
        transforms.Normalize(mean=config["mean"], std=config["std"]),
        transforms.Lambda(lambda x: x * config["rescale"])
    ])
    image = Image.open(image_path).convert("L")
    return transform(image)

In [5]:
class NewOCRDataset(Dataset):
    def __init__(self, custom_data):
        self.custom_data = custom_data
        self.image_paths = list(custom_data.keys())

        with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
            config = yaml.safe_load(f)
        self.char2id = {char: idx for idx, char in config["id2label"].items()}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = preprocess_image(img_path)
        label = self.custom_data[img_path]
        label_ids = [self.char2id.get(c, 0) for c in label]
        return image, torch.tensor(label_ids, dtype=torch.long), len(label_ids)

In [6]:
def train_model(model, dataloader, num_epochs=5):
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch_idx, (images, targets, target_lengths) in enumerate(dataloader):
            target_lengths = torch.tensor(target_lengths, dtype=torch.long)

            optimizer.zero_grad()
            outputs = model(images)
            outputs = outputs.log_softmax(2)

            input_lengths = torch.full((images.size(0),), outputs.size(1), dtype=torch.long)
            loss = criterion(outputs.permute(1, 0, 2), targets, input_lengths, target_lengths)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Avg Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "finetuned.pt")
    print("مدل ذخیره شد: finetuned.pt")

In [17]:
with open("/content/drive/MyDrive/crnn-fa-printed-96-long/model_config.yaml", "r") as f:
        config = yaml.safe_load(f)

model = CRNN(
    n_channels=config["n_channels"],
    map2seq_in_dim=config["map2seq_in_dim"],
    map2seq_out_dim=config["map2seq_out_dim"],
    rnn_dim=config["rnn_dim"],
    num_classes=len(config["id2label"])
)

model.load_state_dict(torch.load("/content/drive/MyDrive/crnn-fa-printed-96-long/model.pt"))
model.eval()

custom_data = {
    "/content/drive/MyDrive/images/ocrImage/code.png": "۹۴۹۰۵۴۹۰۰۸",
    "/content/drive/MyDrive/images/ocrImage/birth.png": "۱۳۶۷/۱۱/۲۳",
    "/content/drive/MyDrive/images/ocrImage/date.png": "۱۴۱۰/۰۳/۲۰",
    '/content/drive/MyDrive/images/ocrImage/father.png': 'اکبر',
    '/content/drive/MyDrive/images/ocrImage/last.png' : 'کرمی',
    "/content/drive/MyDrive/images/ocrImage/name.png": "تیرداد",
}

dataset = NewOCRDataset(custom_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

train_model(model, dataloader, num_epochs=5)

RuntimeError: Error(s) in loading state_dict for CRNN:
	Missing key(s) in state_dict: "cnn.10.0.weight", "cnn.10.0.bias", "cnn.10.1.weight", "cnn.10.1.bias", "cnn.10.1.running_mean", "cnn.10.1.running_var". 
	Unexpected key(s) in state_dict: "cnn.10.conv.weight", "cnn.10.conv.bias", "cnn.10.bn.weight", "cnn.10.bn.bias", "cnn.10.bn.running_mean", "cnn.10.bn.running_var", "cnn.10.bn.num_batches_tracked". 